In [1]:
from get_pts_utils import get_ball_coords, create_label_menu, point_annotator
from pathlib import Path

import os
from typing import List, Tuple

import numpy as np
import pandas as pd
from bioio import BioImage #best way to open .nd2 images
import napari
from skimage.measure import regionprops_table
from numpy.lib.stride_tricks import sliding_window_view
from datetime import datetime
import random
from skimage.io import imsave

In [2]:
IMAGE_DIR = Path("./img_file/")
#IMAGE_DIR = Path("/Users/lecote/Library/CloudStorage/GoogleDrive-lecote@stanford.edu/My Drive/BoxMigration2022-08-16/tempNAS/20260120_hmp-2-hmp-1")

#CSV_DIR = Path("/Users/lecote/Projects/napari/valve_paper/puncta quantification/20260120_hmp-2-hmp-1_csv")
CSV_DIR = Path(IMAGE_DIR.as_posix() + '_csv')
CSV_DIR.mkdir(exist_ok=True)

IM_PROP = dict(channel_axis = 1, 
           blending="additive", 
           name=['DIC','HMP-2', 'HMR-1'], 
           colormap=['gray','magenta','green'], 
           contrast_limits=[(0,65535),(180,3400),(200,14000)])

In [3]:
##go through full folder
metadata_file = CSV_DIR.joinpath('hmp-2_metadata.csv')
file_list = list(IMAGE_DIR.glob("*.nd2"))
for im_path in file_list:
    csv_output_path = CSV_DIR.joinpath(im_path.stem + '.csv')
    if csv_output_path.is_file():
        continue
    print(im_path.stem)
    point_annotator(im_path, 
                IM_PROP, 
                csv_output_path,

                metadata_file,
                labels=['bkgd_1', 
                        'bkgd_2', 
                        'bkgd_3',
                        'ph_1', 
                        'vir_P',
                        'vir_A'],                
                default_notes_values = {'worm' : ['middle'],
                                       'use' : ["yes", "maybe_likely", "maybe_likelynot", "no"], 
                                       'stage' : ['fold-1.5'], 
                                       'red' : ["pos", "het", "mat", "neg"], 
                                       'notes' : ['...'] },
                channels = [1,2],
                radius=[3,1,4,5,7,9],
                keep_values = ['vir_A--3_HMR-1', 'vir_P--3_HMR-1', 'ph_1--1_HMP-2', 'ph_1--3_HMP-2',
                               'vir_A--1_HMR-1', 'vir_P--1_HMR-1',
                               'vir_A--5_HMR-1', 'vir_P--5_HMR-1', 
                               'vir_A--7_HMR-1', 'vir_P--7_HMR-1', 
                               'vir_A--9_HMR-1', 'vir_P--9_HMR-1']
            ) 

20260121_1881_is28.078
Assistant skips harvesting pyclesperanto as it's not installed.
20260126_1884_ctrl.140
20260126_1879_is28.171
20260126_1880_is28.116


In [34]:
from skimage.measure import regionprops_table
from numpy.lib.stride_tricks import sliding_window_view
from datetime import datetime
import random
from skimage.io import imsave

features_list = []
dataframe_list = []
rng = random.Random(42)

In [4]:
#quantify junctional and lateral signal

#list_of_hmp2_img = ['016','017','023','027','031','034','128','129','130','154','170','171','172','173','174','184','185','186','187','188','189','190','191','192','194','196','197','200','201','203','204','205','206','207','208','209','001','001','002','003','004','005','005','006','009','010','140','141','142','143','144','145','146','147','148','150','151','152','153','157','158','159','160','161','162','163','164','165','167','168','169','214','216','217','218','008r','189','200','204']
#shuffled_list = rng.sample(list_of_hmp2_img, k=len(list_of_hmp2_img))

for f in ['171']:
    im_path = list(IMAGE_DIR.glob("*"+f+".nd2"))[0] #find specific image
    IM_PROP = dict(channel_axis = 1, 
               blending="additive", 
               name=['DIC','HMP-2', 'HMR-1'], 
               colormap=['gray','magenta','green'], 
               contrast_limits=[(0,65535),(180,3400),(200,9000)])
    stack = BioImage(im_path)
    dirname, filename = os.path.split(im_path)
    img_name = os.path.join(os.path.split(dirname)[1],filename)
    #stack.dims['Z'][0]

    window_size = 5
    data = stack.data[0][2]
    #create sliding window of max projection
    # Pad to maintain original length, or leave as is for shorter output
    padded_data = np.pad(data, ((2, 2), (0, 0), (0, 0)), mode='edge')
    windowed = sliding_window_view(padded_data, window_shape=window_size, axis=0)
    max_projected_data = np.max(windowed, axis=-1)
    max_projected_data = max_projected_data[np.newaxis, np.newaxis, :] #add dim to get t,c,z,x,y
    
    #open in napari
    viewer = napari.view_image(stack.data, **IM_PROP)
    viewer.layers['DIC'].visible = False
    viewer.layers['HMR-1'].visible = False
    viewer.layers['HMP-2'].visible = False
    max_layer = viewer.add_image(max_projected_data,
                                 channel_axis=1,
               blending="additive", 
               name=['5max_HMR-1'], 
               colormap=['green'], 
               contrast_limits=[(200,9000)])
    #create empty label layer
    gut_layer = viewer.add_labels(np.zeros((stack.dims['Z'][0],512,512), np.uint8))
    gut_layer.brush_size=2
    
    @viewer.bind_key('s')
    def save_measurements(event):
        """Keybinding to save labels and intensities with image name"""
        now = datetime.now()
        props = ['label', 'area', 'mean_intensity', 'max_intensity']
        features_dict = regionprops_table(gut_layer.data, 
                                          intensity_image=max_projected_data[0][0], 
                                          properties=props)
        
        #make junction to lateral ratio
        features_dict['ratio'] = ((features_dict['mean_intensity'][2] - features_dict['mean_intensity'][3])/
                 (features_dict['mean_intensity'][0] - features_dict['mean_intensity'][1]))
        
        features_dict['img_name'] = img_name
        features_dict['worm'] = 'middle'
        #features_dict['worm'] = 'right'
        
        #create dataframe and save data
        features_list.append(features_dict)
        labels_filename = now.strftime("%Y-%m-%d_%H:%M:%S_")+filename+"_"+features_dict['worm']+"_labels.tif"
        labels_savefolder = '/Users/lecote/Projects/napari/valve_paper/puncta quantification/20260213_hmp-2_junctions_layers/'
        to_save = viewer.layers['Labels'].data.astype(np.uint16)
        imsave(labels_savefolder+labels_filename, to_save)
        to_print = [now.strftime("%Y-%m-%d %H:%M:%S"), #time
                    features_dict['img_name'], #img
                    features_dict['worm'], #worm
                    features_dict['mean_intensity'][2], #junctions
                    features_dict['mean_intensity'][3], #cyto_J
                    features_dict['mean_intensity'][0], #lat
                    features_dict['mean_intensity'][1], #cyto_L
                    features_dict['ratio']]
        print(to_print)
        dataframe_list.append(to_print)
    
    viewer.show(block=True) #makes each image pop up separately (blocks code until napari is closed)

In [10]:
df_tocsv = pd.DataFrame(dataframe_list)
df_tocsv.columns = ['time','img','worm','junctions (3)','cyto_J (4)','lateral (1)','cyto_lat (2)','j/l ratio']
df_tocsv.to_csv('./data_files_csv/jtol_v1.csv', mode='w', index=False, header=True)